# Exp8.0 — Local Backbone Temporal-Scale Sweep

This notebook is aggregation-only. It reads finalized CSV/JSON artifacts produced by the Exp8.0 finalizer and does not retrain models or refit probes.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
base = repo / 'notebooks' / 'artifacts' / 'experiment_8_0_local_backbone_tau_sweep' / 'local_backbone_tau_sweep_v1'
manifest = json.loads((base / 'manifest.json').read_text())
runs = pd.read_csv(base / 'method_runs.csv')
summary = pd.read_csv(base / 'method_summary.csv')
contrast_runs = pd.read_csv(base / 'contrast_runs.csv')
contrast_summary = pd.read_csv(base / 'contrast_summary.csv')
activity = pd.read_csv(base / 'activity_summary.csv')
rasters = pd.read_csv(base / 'raster_index.csv')
manifest

## 1. Main method-level comparison

In [ ]:
cols = [
    'architecture',
    'linear_test_ba_mean', 'linear_test_ba_std',
    'lif_test_ba_mean', 'lif_test_ba_std',
    'lif_penalty_mean', 'lif_penalty_std',
    'l1_whole_ba_mean', 'l1_fixed250_ba_mean',
    'l2_whole_ba_mean', 'l2_fixed250_ba_mean',
]
display(summary[cols].copy())

In [ ]:
plot_df = summary.set_index('architecture')
ax = plot_df[['linear_test_ba_mean', 'lif_test_ba_mean']].plot(kind='bar', yerr=plot_df[['linear_test_ba_std', 'lif_test_ba_std']].to_numpy().T, capsize=3)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Native Linear vs same-W output LIF')
ax.set_ylim(0, 1)
plt.tight_layout()

## 2. Representation probes

In [ ]:
probe_cols = ['l1_whole_ba_mean', 'l1_fixed250_ba_mean', 'l2_whole_ba_mean', 'l2_fixed250_ba_mean']
ax = plot_df[probe_cols].plot(kind='bar')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('L1/L2 whole-count and ordered Fixed250 probes')
ax.set_ylim(0, 1)
plt.tight_layout()

In [ ]:
derived = summary[[
    'architecture',
    'l1_temporal_gain_mean', 'l2_temporal_gain_mean',
    'l2_minus_l1_whole_mean', 'l2_minus_l1_fixed250_mean',
]].copy()
display(derived)

## 3. Paired controlled contrasts

In [ ]:
display(contrast_summary)

## 4. Firing/activity diagnostics

In [ ]:
display(activity)

## 5. Per-run raster appendix

Every trained architecture/seed run contributes L1, L2, and same-W output-LIF rasters for the same deterministic representative test sample.

In [ ]:
for architecture in manifest['architectures']:
    for seed in manifest['seeds']:
        rows = rasters[(rasters.architecture == architecture) & (rasters.seed == seed)]
        display(Markdown(f'### {architecture} — seed {seed}'))
        for layer in ['l1', 'l2', 'output']:
            row = rows[rows.layer == layer].iloc[0]
            display(Markdown(f'**{layer.upper()}**'))
            display(Image(filename=str(repo / row.raster_png)))